# PR0504A: Limpieza de datos sobre dataset de lugares famosos

In [1]:
from pyspark.sql import SparkSession

try: 
    spark = (SparkSession.builder.appName("PR0504")
              .master("spark://spark-master:7077")
              .getOrCreate()
            )

    print("SparkSession iniciada correctamente.")
except Exception as e:
    print("Error en la conexion")
    print(e)

sc = spark.sparkContext

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/29 08:40:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession iniciada correctamente.


In [48]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, IntegerType
from pyspark.sql.functions import col, concat, concat_ws, substring, col, lit, upper, lpad, rpad, split, ceil, log10, least, regexp_replace, to_date, date_add, datediff, to_date

places_schema = StructType([
    StructField("Place_Name", StringType(), True),   
    StructField("Country", StringType(), True),     
    StructField("City", StringType(), True),        
    StructField("Annual_Visitors_Millions", DoubleType(), True),
    StructField("Type", StringType(), True),        
    StructField("UNESCO_World_Heritage", StringType(), True),
    StructField("Year_Built", StringType(), True),
    StructField("Entry_Fee_USD", IntegerType(), True),           
    StructField("Best_Visit_Month", StringType(), True),
    StructField("Region", StringType(), True),
    StructField("Tourism_Revenue_Million_USD", LongType(), True),       
    StructField("Average_Visit_Duration_Hours", DoubleType(), True),       
    StructField("Famous_For", StringType(), True)                         
])

df_base = (spark.read
                  .format("csv")
                  .schema(places_schema)
                  .option("header", "true")
                  .load("world_famous_places_2024.csv")
                  .select("Place_Name", "Country", "City", "Type","Year_Built", "Entry_Fee_USD", "Tourism_Revenue_Million_USD", "Average_Visit_Duration_Hours", "Famous_For")
            )

df_base.printSchema()

df_base.show(5, truncate=False)

root
 |-- Place_Name: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Type: string (nullable = true)
 |-- Year_Built: string (nullable = true)
 |-- Entry_Fee_USD: integer (nullable = true)
 |-- Tourism_Revenue_Million_USD: long (nullable = true)
 |-- Average_Visit_Duration_Hours: double (nullable = true)
 |-- Famous_For: string (nullable = true)

+-------------------+-------------+----------------+------------------+----------------+-------------+---------------------------+----------------------------+-------------------------------------------------------+
|Place_Name         |Country      |City            |Type              |Year_Built      |Entry_Fee_USD|Tourism_Revenue_Million_USD|Average_Visit_Duration_Hours|Famous_For                                             |
+-------------------+-------------+----------------+------------------+----------------+-------------+---------------------------+----------------------------+---

## Ejercicio 1: Generación de códigos SKUs

In [41]:
df_feat = df_base.withColumn("SKU_Lugar", 
    concat_ws("_", 
          upper(substring(col("Country"),1, 3)),
          rpad(substring(col("City"), 1, 3), 3, "X"),
          split(col("Type"),"/")[0]
    )
)     
df_feat.show(5)

+-------------------+-------------+----------------+------------------+----------------+-------------+---------------------------+----------------------------+--------------------+--------------------+
|         Place_Name|      Country|            City|              Type|      Year_Built|Entry_Fee_USD|Tourism_Revenue_Million_USD|Average_Visit_Duration_Hours|          Famous_For|           SKU_Lugar|
+-------------------+-------------+----------------+------------------+----------------+-------------+---------------------------+----------------------------+--------------------+--------------------+
|       Eiffel Tower|       France|           Paris|    Monument/Tower|            1889|           35|                         95|                         2.5|Iconic iron latti...|    FRA_Par_Monument|
|       Times Square|United States|   New York City|    Urban Landmark|            1904|            0|                         70|                         1.5|Bright lights, Br...|UNI_New_Urba

## Ejercicio 2: Ajuste de precios y tiempos

In [42]:
df_feat = df_feat.withColumn("Duracion_Techo", ceil(col("Average_Visit_Duration_Hours"))) \
            .withColumn("Log_Ingresos", log10("Tourism_Revenue_Million_USD")) \
            .withColumn("Mejor_Oferta", least("Entry_Fee_USD", lit(20)))

df_feat.show(5)

+-------------------+-------------+----------------+------------------+----------------+-------------+---------------------------+----------------------------+--------------------+--------------------+--------------+------------------+------------+
|         Place_Name|      Country|            City|              Type|      Year_Built|Entry_Fee_USD|Tourism_Revenue_Million_USD|Average_Visit_Duration_Hours|          Famous_For|           SKU_Lugar|Duracion_Techo|      Log_Ingresos|Mejor_Oferta|
+-------------------+-------------+----------------+------------------+----------------+-------------+---------------------------+----------------------------+--------------------+--------------------+--------------+------------------+------------+
|       Eiffel Tower|       France|           Paris|    Monument/Tower|            1889|           35|                         95|                         2.5|Iconic iron latti...|    FRA_Par_Monument|             3|1.9777236052888478|          20|
|   

## Ejercicio 3: Limpieza de texto

In [43]:
df_feat = df_feat.withColumn("Desc_Corta", substring(col("Famous_For"), 1, 15)) \
            .withColumn("Ciudad_Limpia", regexp_replace("City", "New York City", "NYC"))

df_feat.show(5)

+-------------------+-------------+----------------+------------------+----------------+-------------+---------------------------+----------------------------+--------------------+--------------------+--------------+------------------+------------+---------------+----------------+
|         Place_Name|      Country|            City|              Type|      Year_Built|Entry_Fee_USD|Tourism_Revenue_Million_USD|Average_Visit_Duration_Hours|          Famous_For|           SKU_Lugar|Duracion_Techo|      Log_Ingresos|Mejor_Oferta|     Desc_Corta|   Ciudad_Limpia|
+-------------------+-------------+----------------+------------------+----------------+-------------+---------------------------+----------------------------+--------------------+--------------------+--------------+------------------+------------+---------------+----------------+
|       Eiffel Tower|       France|           Paris|    Monument/Tower|            1889|           35|                         95|                        

## Ejercicio 4: Gestión de fechas de campaña

In [52]:
df_feat = df_feat.withColumn("Inicio_Campana", to_date(lit("2023-06-01"))) \
               .withColumn("Fin_De_Campana", date_add(col("Inicio_Campana"), 90)) \
               .withColumn("Dias_Hasta_Fin", datediff(col("Fin_De_Campana"),to_date(concat(col("Year_Built"), lit("-01-01")))))

df_feat.show(5)

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `Fin_De_Camapana` cannot be resolved. Did you mean one of the following? [`Fin_De_Campana`, `Inicio_Campana`, `Place_Name`, `Ciudad_Limpia`, `Desc_Corta`].;
'Project [Place_Name#1550, Country#1551, City#1552, Type#1554, Year_Built#1556, Entry_Fee_USD#1557, Tourism_Revenue_Million_USD#1560L, Average_Visit_Duration_Hours#1561, Famous_For#1562, SKU_Lugar#1632, Duracion_Techo#1693L, Log_Ingresos#1705, Mejor_Oferta#1718, Desc_Corta#1794, Ciudad_Limpia#1809, Inicio_Campana#2136, Fin_De_Campana#2153, datediff('Fin_De_Camapana, to_date(concat(Year_Built#1556, -01-01), None, Some(Etc/UTC), false)) AS Dias_Hasta_Fin#2171]
+- Project [Place_Name#1550, Country#1551, City#1552, Type#1554, Year_Built#1556, Entry_Fee_USD#1557, Tourism_Revenue_Million_USD#1560L, Average_Visit_Duration_Hours#1561, Famous_For#1562, SKU_Lugar#1632, Duracion_Techo#1693L, Log_Ingresos#1705, Mejor_Oferta#1718, Desc_Corta#1794, Ci